In [ ]:
:set -XTemplateHaskell
:set -XQuasiQuotes
import Language.Haskell.TH
import Language.Haskell.TH.Syntax

-- Проверяем что TH доступен
runQ [| 1 + 2 |] >>= print
putStrLn "Setup done."

# 🔮 Метапрограммирование в Haskell

Метапрограммирование — это **написание программ, которые пишут программы**. В Haskell есть несколько мощных инструментов:

| Инструмент | Уровень | Применение |
|------------|---------|------------|
| Template Haskell | Compile-time | Генерация кода, сплайсинг AST |
| GHC Generics | Type-level | Дженерические алгоритмы для ADT |
| Data.Typeable / Data.Data | Runtime | Рефлексия типов |
| QuasiQuotes | Syntax level | Встроенные DSL |

**Предварительные знания:** алгебра типов, монады, классы типов → [TypeAlgebra.ipynb](TypeAlgebra.ipynb), [Monads.ipynb](Monads.ipynb)

**Содержание:**
1. Монада Q и стадии вычисления
2. Цитирование и сплайсинг
3. Template Haskell на практике
4. GHC Generics: дженерические алгоритмы
5. Quasi-quotation
6. Диаграммы

## 1️⃣ Монада Q и стадии вычисления

Template Haskell работает в **двух стадиях**:

```
Стадия 1: Compile-time (время компиляции)
  └── Monada Q :: * -> *
      ├── IO (чтение файлов, переменных окружения)
      ├── Состояние (свежие имена: newName)
      └── Доступ к информации о типах (reify)

Стадия 2: Runtime (время выполнения)
  └── Обычный Haskell-код (сгенерированный TH)
```

**Ключевые типы:**

```haskell
-- Три категории AST-кавычек
type Q a = ...  -- монада компиляции
type Exp = ...  -- выражение (expression)
type Pat = ...  -- паттерн
type Dec = ...  -- объявление (declaration)
type Type = ... -- тип

-- Q-монада похожа на IO, но выполняется при компиляции
runQ :: Q a -> IO a
```

**Операторы сплайсинга:**
- `$(expr)` — подставить выражение из Q Exp
- `[|expr|]` — процитировать выражение → Q Exp
- `$$(expr)` — typed splice (GHC 7.8+)
- `[||expr||]` — typed quotation → Q (TExp a)

## 2️⃣ Цитирование и сплайсинг

**Цитирование (Quotation)** — превращаем Haskell-выражение в его AST:

```haskell
-- Выражение → AST
[| 1 + 2 |]  :: Q Exp
-- ≡ InfixE (Just (LitE (IntegerL 1))) 
--           (VarE (mkName "+"))
--           (Just (LitE (IntegerL 2)))

-- Тип → AST типа
[t| Int -> Bool |] :: Q Type

-- Паттерн → AST паттерна
[p| (x, y) |] :: Q Pat

-- Объявление → AST объявления
[d| f x = x + 1 |] :: [Q Dec]
```

**Сплайсинг (Splicing)** — вставляем AST как код:

```haskell
$(litE (integerL 42))  -- вставляет: 42
$(varE 'show)          -- вставляет: show
```

In [ ]:
runQ [| \x -> x + 1 |] >>= putStrLn . pprint

In [ ]:
runQ [| map (+1) [1,2,3] |] >>= putStrLn . pprint

In [ ]:
runQ [| Just True |] >>= print

In [ ]:
-- Работа с именами
let n1 = mkName "foo"
    n2 = mkName "Data.List.sort"

print n1
print n2

-- 'name -- цитирование имени (value)
-- ''Type -- цитирование имени типа
print 'map       -- Name функции map
print ''Maybe    -- Name типа Maybe
print ''Functor  -- Name класса Functor

## 3️⃣ Template Haskell на практике

### Генерация геттеров

Классический пример: автогенерация функций-геттеров для record-типа.

In [ ]:
:set -XTemplateHaskell

import Language.Haskell.TH

-- Генератор функций id-геттера
-- makeGetter "fieldName" генерирует:
-- getFieldName :: Record -> FieldType
makeGetter :: String -> Q [Dec]
makeGetter fieldName = do
  let getterName = mkName ("get" ++ capitalize fieldName)
      fieldN     = mkName fieldName
  x <- newName "x"
  -- getter r = fieldName r
  let body = NormalB (AppE (VarE fieldN) (VarE x))
      clause = Clause [VarP x] body []
      funDec = FunD getterName [clause]
  return [funDec]
  where
    capitalize [] = []
    capitalize (c:cs) = toEnum (fromEnum c - 32) : cs

-- Посмотрим что генерируется
runQ (makeGetter "name") >>= putStrLn . pprint

In [ ]:
-- Генерация селектора n-ого элемента кортежа
-- sel 3 2 генерирует: \(_, x, _) -> x (для 3-кортежа, 2-ой элемент)
tupleSelector :: Int -> Int -> Q Exp
tupleSelector n i = do
  vars <- mapM (\j -> newName ("x" ++ show j)) [1..n]
  let pat  = TupP (map VarP vars)
      body = VarE (vars !! (i - 1))
  return $ LamE [pat] body

-- Демо: селектор 2-ого из 3
runQ (tupleSelector 3 2) >>= putStrLn . pprint
putStrLn "---"
-- селектор 1-ого из 4
runQ (tupleSelector 4 1) >>= putStrLn . pprint

In [ ]:
import Data.List (sort)

-- Цитирование имён: ' для значений, '' для типов
putStrLn $ "Name для Maybe: " ++ show ''Maybe
putStrLn $ "Name для map:   " ++ show 'map
putStrLn $ "Name для sort:  " ++ show 'sort
putStrLn $ "Name для Eq:    " ++ show ''Eq

-- nameBase: имя без модуля, nameModule: имя модуля
let n = 'map
putStrLn $ "nameBase 'map:   " ++ nameBase n
putStrLn $ "nameModule 'map: " ++ show (nameModule n)

-- Примечание: reify работает в compile-time (в Q-монаде),
-- но не в интерактивном REPL из-за staging restrictions

## 4️⃣ GHC Generics: дженерические алгоритмы

**GHC.Generics** позволяет писать алгоритмы, которые работают для **любого ADT**
без дублирования кода.

### Структура Generic

Любой ADT представляется через комбинаторы:

| Комбинатор | Значение | Пример |
|------------|----------|--------|
| `V1` | Пустой тип (Void) | Нет конструкторов |
| `U1` | Единица (Unit) | Конструктор без полей |
| `a :+: b` | Сумма | Either-подобные ADT |
| `a :*: b` | Произведение | Пары/записи |
| `K1 i c` | Константа (поле) | Поле типа c |
| `M1 i c f` | Метаданные | Имена, модули |
| `Rec1 f` | Рекурсивное поле | f a в полиморфных типах |

```haskell
-- Для data Tree a = Leaf | Node a (Tree a) (Tree a)
-- Generic (Tree a) даёт:
type Rep (Tree a) = 
  M1 D ('MetaData "Tree" ...) (
    M1 C ('MetaCons "Leaf" ...) U1
    :+:
    M1 C ('MetaCons "Node" ...) (
      M1 S ('MetaSel ...) (K1 R a)
      :*: M1 S ('MetaSel ...) (K1 R (Tree a))
      :*: M1 S ('MetaSel ...) (K1 R (Tree a))
    )
  )
```

In [ ]:
:set -XDeriveGeneric
:set -XDefaultSignatures
:set -XFlexibleContexts
:set -XFlexibleInstances
:set -XTypeSynonymInstances
:set -XScopedTypeVariables
import GHC.Generics

-- Простой тип с Generic
data Shape
  = Circle  { radius :: Double }
  | Rect    { width :: Double, height :: Double }
  | Point
  deriving (Show, Generic)

-- Посмотрим на Rep
:kind! Rep Shape

In [ ]:
-- Дженерический подсчёт конструкторов
class GCountCtors f where
  gCountCtors :: f a -> Int

instance GCountCtors V1 where
  gCountCtors _ = 0

instance GCountCtors U1 where
  gCountCtors _ = 1

instance (GCountCtors f, GCountCtors g) => GCountCtors (f :+: g) where
  gCountCtors (L1 x) = gCountCtors x
  gCountCtors (R1 x) = gCountCtors x

instance (GCountCtors f, GCountCtors g) => GCountCtors (f :*: g) where
  gCountCtors (x :*: _) = gCountCtors x

instance GCountCtors (K1 i c) where
  gCountCtors _ = 1

instance GCountCtors f => GCountCtors (M1 i c f) where
  gCountCtors (M1 x) = gCountCtors x

countCtors :: (Generic a, GCountCtors (Rep a)) => a -> Int
countCtors x = gCountCtors (from x)

-- Тест
putStrLn $ "Circle конструктор: " ++ show (countCtors (Circle 5.0))
putStrLn $ "Rect конструктор:   " ++ show (countCtors (Rect 3 4))
putStrLn $ "Point конструктор:  " ++ show (countCtors Point)

In [ ]:
-- Дженерическая сериализация в список пар (имя -> значение)
-- Упрощённая версия: только числовые поля

class GFields f where
  gFields :: f a -> [(String, String)]

instance GFields U1 where
  gFields _ = []

instance GFields V1 where
  gFields _ = []

instance (GFields f, GFields g) => GFields (f :+: g) where
  gFields (L1 x) = gFields x
  gFields (R1 x) = gFields x

instance (GFields f, GFields g) => GFields (f :*: g) where
  gFields (x :*: y) = gFields x ++ gFields y

instance Show c => GFields (K1 i c) where
  gFields (K1 x) = [("_", show x)]

instance (GFields f, Selector s) => GFields (M1 S s f) where
  gFields m@(M1 x) = 
    let name = selName m
        vals = gFields x
    in if null name then vals else [(name, snd (head vals)) | not (null vals)]

instance GFields f => GFields (M1 D c f) where
  gFields (M1 x) = gFields x

instance GFields f => GFields (M1 C c f) where
  gFields (M1 x) = gFields x

toFields :: (Generic a, GFields (Rep a)) => a -> [(String, String)]
toFields = gFields . from

-- Тест
putStrLn "Circle 5.0:"
mapM_ print (toFields (Circle 5.0))
putStrLn "Rect 3.0 4.0:"
mapM_ print (toFields (Rect 3.0 4.0))

In [ ]:
-- Дженерическое равенство (как работает deriving Eq)
class GEq f where
  geq :: f a -> f a -> Bool

instance GEq U1 where
  geq U1 U1 = True

instance GEq V1 where
  geq _ _ = True

instance Eq c => GEq (K1 i c) where
  geq (K1 x) (K1 y) = x == y

instance GEq f => GEq (M1 i c f) where
  geq (M1 x) (M1 y) = geq x y

instance (GEq f, GEq g) => GEq (f :+: g) where
  geq (L1 x) (L1 y) = geq x y
  geq (R1 x) (R1 y) = geq x y
  geq _ _            = False

instance (GEq f, GEq g) => GEq (f :*: g) where
  geq (x1 :*: y1) (x2 :*: y2) = geq x1 x2 && geq y1 y2

genericEq :: (Generic a, GEq (Rep a)) => a -> a -> Bool
genericEq x y = geq (from x) (from y)

-- Тест
putStrLn $ "Circle 5 == Circle 5: " ++ show (genericEq (Circle 5) (Circle 5))
putStrLn $ "Circle 5 == Circle 6: " ++ show (genericEq (Circle 5) (Circle 6))
putStrLn $ "Rect 3 4 == Rect 3 4: " ++ show (genericEq (Rect 3 4) (Rect 3 4))
putStrLn $ "Point == Point:        " ++ show (genericEq Point Point)

## 5️⃣ Quasi-quotation

**QuasiQuoters** — позволяют встраивать пользовательские DSL прямо в Haskell-синтаксис.

```haskell
-- Синтаксис: [quoterName| ... текст ... |]

-- Примеры из библиотек:
[r| raw string \n не escaping |]      -- raw-строки
[sql| SELECT * FROM users |]          -- встроенный SQL
[html| <div>Hello</div> |]            -- HTML шаблоны
[py| x = 1; y = x + 2 |]             -- inline Python
```

**Структура квазицитатора:**

```haskell
myQQ :: QuasiQuoter
myQQ = QuasiQuoter
  { quoteExp  = \s -> [| parseMyDSL s |]   -- в позиции выражения
  , quotePat  = \s -> ...                   -- в позиции паттерна
  , quoteType = \s -> ...                   -- в позиции типа
  , quoteDec  = \s -> ...                   -- в позиции объявления
  }
```

In [ ]:
:set -XQuasiQuotes
import Language.Haskell.TH.Quote

-- Простой квазицитатор: разбивает строку по пробелам в список
wordsQQ :: QuasiQuoter
wordsQQ = QuasiQuoter
  { quoteExp = \s -> lift (words s)
  , quotePat  = error "wordsQQ: no pattern"
  , quoteType = error "wordsQQ: no type"
  , quoteDec  = error "wordsQQ: no decl"
  }

-- Квазицитатор для целых чисел через запятую
numsQQ :: QuasiQuoter
numsQQ = QuasiQuoter
  { quoteExp = \s -> lift (map (read :: String -> Int) (words (map (\c -> if c == ',' then ' ' else c) s)))
  , quotePat  = error "numsQQ: no pattern"
  , quoteType = error "numsQQ: no type"
  , quoteDec  = error "numsQQ: no decl"
  }
putStrLn "QuasiQuoters defined"

In [ ]:
-- QuasiQuoters, определённые в REPL, нельзя использовать в том же сеансе
-- (нужна компиляция). Но можно использовать встроенные:
-- Пример с multiline strings через heredoc (если доступно)

-- Демонстрируем идею вручную:
let applyWordsQQ s = words s
let applyNumsQQ s = map (read :: String -> Int) . words $ map (\c -> if c == ',' then ' ' else c) s

print (applyWordsQQ "hello world foo bar")
print (applyNumsQQ "1, 2, 3, 42, 100")

-- В реальном проекте: [wordsQQ| hello world |] раскрывается в compile-time
-- Требует: {-# LANGUAGE QuasiQuotes #-} и импорт модуля с QQ

## 6️⃣ Data.Typeable и Data.Data

Для **runtime-рефлексии** (не compile-time) есть `Typeable` и `Data`:

```haskell
-- Typeable: динамические типы
typeRep :: Typeable a => proxy a -> TypeRep
cast    :: (Typeable a, Typeable b) => a -> Maybe b

-- Data: структурная рефлексия (как Generics, но runtime)
gfoldl  :: Data d => (forall a b. Data a => c (a -> b) -> a -> c b) -> ...
gmapT   :: Data a => (forall b. Data b => b -> b) -> a -> a
toConstr :: Data a => a -> Constr
```

In [ ]:
import Data.Typeable

-- typeOf возвращает TypeRep
print (typeOf (42 :: Int))
print (typeOf "hello")
print (typeOf (Just True))
print (typeOf (id :: Int -> Int))

-- cast: безопасное приведение типов
let x = 42 :: Int
    y = cast x :: Maybe Int
    z = cast x :: Maybe Bool  -- Nothing: несовместимые типы
putStrLn $ "cast Int -> Int:  " ++ show y
putStrLn $ "cast Int -> Bool: " ++ show z

In [ ]:
import Data.Data

data Person = Person { pName :: String, pAge :: Int }
  deriving (Show, Data, Typeable)

let alice = Person "Alice" 30

-- Структурная информация
putStrLn $ "Конструктор: " ++ show (toConstr alice)
putStrLn $ "Тип данных:  " ++ show (dataTypeOf alice)
putStrLn $ "Все конструкторы: " ++ show (dataTypeConstrs (dataTypeOf alice))

-- gunfold: создание значений
-- constrFields: поля конструктора
putStrLn $ "Поля Person: " ++ show (constrFields (toConstr alice))
putStrLn $ "Индекс конструктора: " ++ show (constrIndex (toConstr alice))

## Диаграммы: стадии TH и структура Generic

Ниже — схема стадий Template Haskell и структура представления ADT в GHC.Generics:

- Как `Q`-монада трансформирует код во время компиляции
- Как GHC.Generics представляет произвольные ADT

![Стадии Template Haskell](../diagrams/meta/mp_th_stages.svg)

![Структура GHC.Generics](../diagrams/meta/mp_generics.svg)

## Итог

| Инструмент | Когда | Пример использования |
|------------|-------|---------------------|
| Template Haskell | Compile-time | Генерация lens, aeson, persistent |
| GHC.Generics | Compile/type-level | JSON, Binary, NFData, Eq, Ord, Show |
| QuasiQuotes | Compile-time | Встроенные DSL (SQL, HTML, regex) |
| Typeable | Runtime | Динамическая типизация, cast |
| Data.Data | Runtime | Структурный обход, Scrap Your Boilerplate |

**Ключевые идеи:**
- TH работает в монаде `Q`, выполняется **при компиляции**
- Generics превращают ADT в **алгебраическое представление** (U1, :+:, :*:, K1)
- QuasiQuotes = произвольный парсер → Haskell AST
- `Typeable` / `Data` — runtime-аналоги Generics

🎯 **Практическое применение:** библиотеки `aeson`, `lens`, `persistent`, `servant` используют TH или Generics для устранения шаблонного кода.

---

**← Предыдущий:** [Сопряжения](Adjunctions.ipynb)  |  **Следующий →** [Конкурентность](Concurrency.ipynb)
